In [ ]:
# py -m pip install git+https://github.com/pbtamarona/OpenPyTEA
from openpytea.equipment import *
from openpytea.plant import *
from openpytea.analysis import *
from openpytea.plotting import *

In [ ]:
#check version
from importlib.metadata import version
print(version("openpytea"))

In [ ]:
#Generic settings for plots

import scienceplots
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['axes.grid'] = False

# ticks
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'

mpl.rcParams['xtick.top'] = True
mpl.rcParams['xtick.bottom'] = True
mpl.rcParams['ytick.left'] = True
mpl.rcParams['ytick.right'] = True

mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

mpl.rcParams['xtick.major.top'] = True
mpl.rcParams['xtick.major.bottom'] = True
mpl.rcParams['ytick.major.left'] = True
mpl.rcParams['ytick.major.right'] = True

mpl.rcParams['xtick.minor.top'] = True
mpl.rcParams['xtick.minor.bottom'] = True
mpl.rcParams['ytick.minor.left'] = True
mpl.rcParams['ytick.minor.right'] = True

In [ ]:
#OPEX Inputs

country, region = 'Netherlands', 'None'
interest_rate, project_lifetime, plant_utilization = 0.09, 20, 0.95
h2_prod = 100_000  # Daily hydrogen production (kg/day)
tax_rate = 0.258

# Operator hourly rate ($/h)
operator_rate, operator_std, operator_max, operator_min = 39.04, 5.03548, 55, 23.43

# Electricity price ($/kWh)
elecs_price, elecs_std, elecs_max, elecs_min = 0.125, 0.0227273, 0.5, 0.05

# Natural gas price ($/kg)
nat_gas_price, nat_gas_std, nat_gas_max, nat_gas_min = 0.8646, 0.21615, 4.884, 0.30261

# Cooling water price ($/MJ)
cool_water_price, cool_water_std, cool_water_max, cool_water_min = 2.4592e-4, 4.815e-5, 4e-4, 1e-5

# CO2 tax ($/kg CO2)
co2_tax, co2_std, co2_max, co2_min = 0.10201, 0.0258794, 0.19, 0

# Catalyst price ($/kg)
cat_price, cat_std, cat_max, cat_min = 16.5, 6.25, 30, 5


In [ ]:
#PSA Sizing

import numpy as np

def calculate_psa_vessel_thickness(vessel_diameter, diameter_threshold=3.5):
    # Minimum practical wall thickness from Towler & Sinnott
    if vessel_diameter <= 1.0:
        minimum_thickness_mm = 5.0
    elif vessel_diameter <= 2.0:
        minimum_thickness_mm = 7.0
    elif vessel_diameter <= 2.5:
        minimum_thickness_mm = 9.0
    elif vessel_diameter <= 3.0:
        minimum_thickness_mm = 10.0
    else:
        minimum_thickness_mm = 12.0

    # Additional thickness margin
    if vessel_diameter <= diameter_threshold:
        extra_thickness_mm = 5.0
    else:
        extra_thickness_mm = 10.0

    return minimum_thickness_mm + extra_thickness_mm


def calculate_psa_vessel_dimensions(adsorbent_mass_per_bed, rho_bulk=795.0, inert_fraction=0.20, aspect_ratio=4.0):
    # Adsorbent volume per bed [m3]
    adsorbent_volume = adsorbent_mass_per_bed / rho_bulk

    # Total internal vessel volume between tangent lines [m3]
    vessel_volume = adsorbent_volume / (1 - inert_fraction)

    # Vessel geometry
    vessel_diameter = (4 * vessel_volume / (np.pi * aspect_ratio)) ** (1 / 3)
    vessel_height = aspect_ratio * vessel_diameter

    return {
        "adsorbent_volume": adsorbent_volume,
        "vessel_volume": vessel_volume,
        "vessel_diameter": vessel_diameter,
        "vessel_height": vessel_height,
        "aspect_ratio": aspect_ratio
    }


def calculate_psa_shell_mass(vessel_diameter, vessel_height, steel_density=7850.0, diameter_threshold=3.5):
    # Wall thickness from vessel diameter
    thickness_mm = calculate_psa_vessel_thickness(vessel_diameter, diameter_threshold=diameter_threshold)
    thickness = thickness_mm / 1000  # m

    # Internal and external diameters
    D_inner = vessel_diameter
    D_outer = D_inner + 2 * thickness

    # Steel volume of cylindrical shell [m3]
    steel_volume = np.pi / 4 * (D_outer**2 - D_inner**2) * vessel_height

    # Shell mass [kg]
    shell_mass = steel_volume * steel_density

    return {
        "wall_thickness_mm": thickness_mm,
        "steel_volume": steel_volume,
        "shell_mass": shell_mass
    }


def calculate_psa_design(df, p_ads_bar=15.0, p_des_bar=1.01, adsorption_time=300.0, bed_loading_fraction=0.85, min_trains=2, n_beds_per_train=4, h2_purity=0.995, product_mass_flow=100_000, rho_bulk=795.0, inert_fraction=0.20, min_aspect_ratio=4.0, max_aspect_ratio=6.0, aspect_ratio_step=0.5, max_diameter=3.5, max_trains=15):


    psa_feed_total = df["PSA Mole Feed [kmol/hr]"].iloc[0]
    ch4_feed_total = df["CH4 Reactor Outlet [kmol/h]"].iloc[0]

    MW_H2 = 2.01588    # kg/kmol
    MW_CH4 = 16.04246  # kg/kmol


    product_mass_total = product_mass_flow / 24  # kg/h

    # Product purity
    MW_product = h2_purity * MW_H2 + (1 - h2_purity) * MW_CH4

    # Total product molar flow and component molar flows [kmol/h]
    total_product_total = product_mass_total / MW_product
    h2_product_total = total_product_total * h2_purity
    ch4_product_total = total_product_total * (1 - h2_purity)

    # CH4 / ZEOLITE 5A ISOTHERM - Towler & Sinnott, Example 16.1
    pressure_data = np.array([0.0, 1.5, 2.5, 4.5, 6.5, 9.0, 12.5, 16.5, 19.5])
    loading_data = np.array([0.0, 0.4, 0.7, 1.0, 1.25, 1.45, 1.60, 1.70, 1.75])  # mmol/g = mol/kg

    bar_to_atm = 0.986923
    valid_design = False
    diameter_fallback = False

    # Number of trains and H/D ratio
    for n_trains in range(min_trains, max_trains + 1):

        # Flow handled by one PSA train
        psa_feed = psa_feed_total / n_trains
        ch4_feed = ch4_feed_total / n_trains
        h2_feed = psa_feed - ch4_feed

        # Product handled by one PSA train
        total_product = total_product_total / n_trains
        h2_product = h2_product_total / n_trains
        ch4_product = ch4_product_total / n_trains


        h2_recovery = h2_product / h2_feed

        # PSA off-gas
        h2_offgas = h2_feed - h2_product
        ch4_offgas = ch4_feed - ch4_product
        total_offgas = h2_offgas + ch4_offgas

        if h2_offgas < 0 or ch4_offgas < 0 or total_offgas <= 0:
            raise ValueError("Calculated PSA outlet flows are not physically consistent.")

        # CH4 mole fractions
        y_ch4_feed = ch4_feed / psa_feed
        y_ch4_offgas = ch4_offgas / total_offgas

        # CH4 partial pressures [atm]
        p_ch4_ads = p_ads_bar * y_ch4_feed * bar_to_atm
        p_ch4_des = p_des_bar * y_ch4_offgas * bar_to_atm

        if p_ch4_ads > pressure_data[-1]:
            raise ValueError("Adsorption CH4 partial pressure exceeds isotherm range.")

        if p_ch4_des > pressure_data[-1]:
            raise ValueError("Desorption CH4 partial pressure exceeds isotherm range.")

        # Equilibrium loadings
        q_ads = np.interp(p_ch4_ads, pressure_data, loading_data)
        q_des = np.interp(p_ch4_des, pressure_data, loading_data)
        working_capacity = q_ads - q_des

        if working_capacity <= 0:
            raise ValueError("PSA working capacity is zero or negative.")

        # ADSORBENT MASS PER BED - Towler Eq. 16.2
        ch4_removed = ch4_feed - ch4_product  # kmol/h
        ch4_removed_mol_s = ch4_removed * 1000 / 3600
        adsorbent_mass_per_bed = ch4_removed_mol_s * adsorption_time / (working_capacity * bed_loading_fraction)

        # Discrete H/D ratios
        aspect_ratios = np.arange(min_aspect_ratio, max_aspect_ratio + aspect_ratio_step / 2, aspect_ratio_step)

        for aspect_ratio in aspect_ratios:
            vessel = calculate_psa_vessel_dimensions(adsorbent_mass_per_bed, rho_bulk=rho_bulk, inert_fraction=inert_fraction, aspect_ratio=aspect_ratio)

            if vessel["vessel_diameter"] <= max_diameter:
                valid_design = True
                break

        # Fallback at max number of trains
        if not valid_design and n_trains == max_trains:
            aspect_ratio = max_aspect_ratio
            vessel = calculate_psa_vessel_dimensions(adsorbent_mass_per_bed, rho_bulk=rho_bulk, inert_fraction=inert_fraction, aspect_ratio=aspect_ratio)

            valid_design = True
            diameter_fallback = True

        if valid_design:
            break

    if not valid_design:
        raise ValueError("No valid PSA configuration could be determined.")

    # Total number of vessels and adsorbent mass
    n_vessels = n_trains * n_beds_per_train
    total_adsorbent_mass = adsorbent_mass_per_bed * n_vessels

    # STEEL VOLUME AND SHELL MASS
    shell = calculate_psa_shell_mass(vessel["vessel_diameter"], vessel["vessel_height"], diameter_threshold=max_diameter)
    total_shell_mass = shell["shell_mass"] * n_vessels

    return {
        "h2_recovery": h2_recovery,
        "total_product_molar_flow": total_product_total,
        "h2_product_molar_flow": h2_product_total,
        "ch4_product_molar_flow": ch4_product_total,
        "adsorbent_mass_per_bed": adsorbent_mass_per_bed,
        "total_adsorbent_mass": total_adsorbent_mass,
        "vessel_volume": vessel["vessel_volume"],
        "vessel_diameter": vessel["vessel_diameter"],
        "vessel_height": vessel["vessel_height"],
        "aspect_ratio": vessel["aspect_ratio"],
        "wall_thickness_mm": shell["wall_thickness_mm"],
        "steel_volume_per_vessel": shell["steel_volume"],
        "shell_mass_per_vessel": shell["shell_mass"],
        "total_shell_mass": total_shell_mass,
        "n_trains": n_trains,
        "n_beds_per_train": n_beds_per_train,
        "n_vessels": n_vessels,
        "diameter_fallback": diameter_fallback
    }

In [ ]:
def run_tea_pfr(csv_path):

    df = pd.read_csv(csv_path)

    # START OF EQUIPMENT DEFINITION
    comp_1_pwr = df["Comp-1 Power [kW]"].iloc[0]  # kW
    comp_2_pwr = df["Comp-2 Power [kW]"].iloc[0]  # kW

    hx1_area = df["HeatX-1 Area [m2]"].iloc[0]
    hx2_area = df["HeatX-2 Area [m2]"].iloc[0]

    cooler1_area = df["Cooler-1 Area [m2]"].iloc[0]
    cooler2_area = df["Cooler-2 Area [m2]"].iloc[0]

    reactor_volume = df["Reactor Volume [m3]"].iloc[0]  # m3

    total_heating = (
        df["Heater-1 Duty [MW]"].iloc[0]
        + df["Heater-2 Duty [MW]"].iloc[0]
        + df["Reactor Duty [MW]"].iloc[0]
    ) * 1000  # Total thermal duty in kW

    cyclone_rate = df["Cyclone Volumetric [m3/s]"].iloc[0]


    # Electrical-to-mechanical efficiency of compressor drives.
    elec_mecc_eff = 0.90

    # Electrical power absorbed by the compressor drives.
    mtr_1_pwr = comp_1_pwr / elec_mecc_eff
    mtr_2_pwr = comp_2_pwr / elec_mecc_eff


    # COMP-1 uses the Seider turbo-blower correlation; requires horsepower and already includes its motor.
    comp_1_pwr_hp = comp_1_pwr * 1.34102
    comp_1 = Equipment('COMP-1', comp_1_pwr_hp, "Fluids", "Compressors & blowers", "Centrifugal (turbo) blower", "Carbon steel")

    # COMP-2 uses the Towler centrifugal-compressor correlation.
    comp_2 = Equipment('COMP-2', comp_2_pwr, "Fluids", "Compressors, fans, & blowers", "Compressor, centrifugal", "Carbon steel")

    mtr_2 = Equipment('MTR-2', comp_2_pwr, "Electrical", "Motors & generators", "Explosion-proof motor", "Carbon steel")

    #electric heating efficiency
    react_heat_eff = 0.90

    # Electric power required to supply the total thermal duty.
    electric_heating_power = total_heating / react_heat_eff  # kW

    #Electric arc furnace
    furnace = Equipment('FURNACE', electric_heating_power / 1000, "Fluids", "Boilers, heaters, & furnaces", "Electric arc furnace", "Inconel")

    #heat exchangers
    hx_1 = Equipment('HX-1', hx1_area, "Fluids", "Heat exchangers", "Floating head shell & tube", "321 stainless steel")
    hx_2 = Equipment('HX-2', max(hx2_area, 10), "Fluids", "Heat exchangers", "Fixed tube", "Carbon steel")

    #coolers
    cooler_1 = Equipment('COOLER-1', cooler1_area, "Fluids", "Heat exchangers", "Fixed tube", "Carbon steel")
    cooler_2 = Equipment('COOLER-2', cooler2_area, "Fluids", "Heat exchangers", "Fixed tube", "Carbon steel")

    #reactor
    NiMP_PFR_reactor = Equipment('NiMP_reactor', reactor_volume, "Mixed", "Reactors", "Indirect fluidized-bed", "321 stainless steel")

    cyclone = Equipment('CYCLONE', cyclone_rate, "Mixed", "Dust collectors", "Gas multi-cyclone", "321 stainless steel")

    # PSA
    psa_design = calculate_psa_design(df, p_ads_bar=15.0, p_des_bar=1.01, min_trains=2, n_beds_per_train=4)

    psa_shell_mass = psa_design["shell_mass_per_vessel"]
    psa_adsorbent_mass_total = psa_design["total_adsorbent_mass"]
    n_psa_vessels = psa_design["n_vessels"]

    # PSA vessels
    psa_vessel = Equipment(name="PSA-VESSEL", param=psa_shell_mass, process_type="Fluids", category="Pressure vessels", type="Vertical CS", material="Carbon steel", num_units=n_psa_vessels)

    # Zeolite 5A
    psa_adsorbent_volume_total_m3 = psa_adsorbent_mass_total / 795.0
    psa_adsorbent_volume_total_ft3 = psa_adsorbent_volume_total_m3 * 35.3146667

    psa_zeolite = Equipment(name="PSA-ZEOLITE-5A", param=psa_adsorbent_volume_total_ft3, process_type="Solids", category="Packings & adsorbents", type="Molecular sieves")

    # Complete PSA unit
    psa = CompositeEquipment(name="PSA", process_type="Fluids", components=[psa_vessel, psa_zeolite], category="Adsorption", type="Pressure-swing adsorber")

    NiMP_PFR_equipment = [
        comp_1,
        comp_2, mtr_2,
        hx_1, hx_2,
        cooler_1, cooler_2,
        NiMP_PFR_reactor,
        furnace, cyclone,
        psa
    ]


    # START OF PLANT
    methane_feed = df["CH4 Feed [kg/h]"].iloc[0]  # kg/h
    coolwater_total = df["Cool Water Total [kJ/s]"].iloc[0]
    carbon_prod = df["C Prod [kg/h]"].iloc[0]
    catalyst_flow = df["Cat Rate [kg/h]"].iloc[0]
    ch4_reactor_in = df["CH4 Reactor Feed [kmol/h]"].iloc[0]
    ch4_reactor_out = df["CH4 Reactor Outlet [kmol/h]"].iloc[0]

    residence_time = df["Cat Res. Time [s]"].iloc[0]
    deactivation_factor = df["Reactor Deactivation"].iloc[0]


    # Configuration for the NiMP plant.
    config = {
        'plant_name': 'PFR',
        'process_type': 'Mixed',
        'country': country,
        'region': region,
        'equipment': NiMP_PFR_equipment,
        'interest_rate': interest_rate,
        'tax_rate': tax_rate,
        'project_lifetime': project_lifetime,
        'plant_utilization': plant_utilization,

        'operator_hourly_rate': {
            'rate': operator_rate,
            'std': operator_std,
            'max': operator_max,
            'min': operator_min
        },

        'plant_products': {
            'hydrogen': {
                'production': h2_prod,
                'price': 5.0
            },

            'carbon': {
                'production': (carbon_prod + catalyst_flow) * 0.5 * 24,
                'price': 1,
                'std': 0.5,
                'max': 6,
                'min': 0
            }
        },

        'variable_opex_inputs': {
            'electricity': {
                'consumption': (mtr_1_pwr + mtr_2_pwr + electric_heating_power) * 24,
                'price': elecs_price,
                'std': elecs_std,
                'max': elecs_max,
                'min': elecs_min
            },

            'methane_feed': {
                'consumption': methane_feed * 24,
                'price': nat_gas_price,
                'std': nat_gas_std,
                'max': nat_gas_max,
                'min': nat_gas_min
            },

            'catalyst_feed': {
                'consumption': catalyst_flow * 24,
                'price': cat_price,
                'std': cat_std,
                'max': cat_max,
                'min': cat_min
            },

            'cooling_water': {
                'consumption': coolwater_total * 3600 * 24 / 1000,
                'price': cool_water_price,
                'std': cool_water_std,
                'max': cool_water_max,
                'min': cool_water_min
            },

            'carbon_disposal_fee': {
                'consumption': (carbon_prod + catalyst_flow) * 0.5 * 24,
                'price': 0.5,
                'std': 0.5 / 2,
                'max': 1,
                'min': 0
            }
        }
    }


    NiMP_PFR_plant = Plant(config)

    ch4_conversion = (ch4_reactor_in - ch4_reactor_out) * 100 / ch4_reactor_in


    # START OF ECONOMIC ANALYSIS
    capex = NiMP_PFR_plant.calculate_fixed_capital(print_results=False)
    isbl = NiMP_PFR_plant.calculate_isbl(print_results=False)
    fixed_opex = NiMP_PFR_plant.calculate_fixed_opex(print_results=False)
    variable_opex = NiMP_PFR_plant.calculate_variable_opex(print_results=False)
    revenue = NiMP_PFR_plant.calculate_revenue(print_results=False)

    cash_flow_styler = NiMP_PFR_plant.calculate_cash_flow(print_results=True)
    cash_flow_df = cash_flow_styler.data

    npv = NiMP_PFR_plant.calculate_npv(print_results=False)
    pbt = NiMP_PFR_plant.calculate_payback_time(print_results=False)
    pbt_add = NiMP_PFR_plant.calculate_payback_time(print_results=False, additional_capex=True)
    lcoh = NiMP_PFR_plant.calculate_levelized_cost(print_results=False)
    roi = NiMP_PFR_plant.calculate_roi(print_results=False)
    roi_add = NiMP_PFR_plant.calculate_roi(print_results=False, additional_capex=True)
    irr = NiMP_PFR_plant.calculate_irr(print_results=False)


    return {
        "plant_pfr": NiMP_PFR_plant,
        "capex_pfr": capex,
        "isbl_pfr": isbl,
        "fixed_opex_pfr": fixed_opex,
        "variable_opex_pfr": variable_opex,
        "revenue_pfr": revenue,
        "cash_flow_pfr": cash_flow_df,
        "npv_pfr": npv,
        "pbt_pfr": pbt,
        "pbt_add_pfr": pbt_add,
        "lcoh_pfr": lcoh,
        "roi_pfr": roi,
        "roi_add_pfr": roi_add,
        "irr_pfr": irr,
        "catalyst_rate_pfr": catalyst_flow,
        "deac_factor_pfr": deactivation_factor,
        "conversion_pfr": ch4_conversion,
        "res_time_pfr": residence_time
    }

In [ ]:
#Similar to PFR
def run_tea_cstr(csv_path):

    df = pd.read_csv(csv_path)

    # START OF EQUIPMENT DEFINITION
    comp_1_pwr = df["Comp-1 Power [kW]"].iloc[0]  # kW
    comp_2_pwr = df["Comp-2 Power [kW]"].iloc[0]  # kW

    hx1_area = df["HeatX-1 Area [m2]"].iloc[0]
    hx2_area = df["HeatX-2 Area [m2]"].iloc[0]

    cooler1_area = df["Cooler-1 Area [m2]"].iloc[0]
    cooler2_area = df["Cooler-2 Area [m2]"].iloc[0]

    reactor_volume = df["Reactor Volume [m3]"].iloc[0]  # m3

    total_heating = (
        df["Heater-1 Duty [MW]"].iloc[0]
        + df["Heater-2 Duty [MW]"].iloc[0]
        + df["Reactor Duty [MW]"].iloc[0]
    ) * 1000  # Total thermal duty in kW

    cyclone_rate = df["Cyclone Volumetric [m3/s]"].iloc[0]


    # Electrical-to-mechanical efficiency of compressor drives.
    elec_mecc_eff = 0.90


    mtr_1_pwr = comp_1_pwr / elec_mecc_eff
    mtr_2_pwr = comp_2_pwr / elec_mecc_eff

    # COMP-1 uses the Seider turbo-blower correlation; requires horsepower and already includes its motor.
    comp_1_pwr_hp = comp_1_pwr * 1.34102
    comp_1 = Equipment('COMP-1', comp_1_pwr_hp, "Fluids", "Compressors & blowers", "Centrifugal (turbo) blower", "Carbon steel")

    # COMP-2 uses the Towler centrifugal-compressor correlation.
    comp_2 = Equipment('COMP-2', comp_2_pwr, "Fluids", "Compressors, fans, & blowers", "Compressor, centrifugal", "Carbon steel")

    mtr_2 = Equipment('MTR-2', comp_2_pwr, "Electrical", "Motors & generators", "Explosion-proof motor", "Carbon steel")

    #electric heating efficiency
    react_heat_eff = 0.90

    # Electric power required to supply the total thermal duty.
    electric_heating_power = total_heating / react_heat_eff  # kW

    furnace = Equipment('FURNACE', electric_heating_power / 1000, "Fluids", "Boilers, heaters, & furnaces", "Electric arc furnace", "Inconel")

    hx_1 = Equipment('HX-1', hx1_area, "Fluids", "Heat exchangers", "Floating head shell & tube", "321 stainless steel")
    hx_2 = Equipment('HX-2', max(hx2_area, 10), "Fluids", "Heat exchangers", "Fixed tube", "Carbon steel")

    cooler_1 = Equipment('COOLER-1', cooler1_area, "Fluids", "Heat exchangers", "Fixed tube", "Carbon steel")
    cooler_2 = Equipment('COOLER-2', cooler2_area, "Fluids", "Heat exchangers", "Fixed tube", "Carbon steel")

    NiMP_CSTR_reactor = Equipment('NiMP_reactor', reactor_volume, "Mixed", "Reactors", "Indirect fluidized-bed", "321 stainless steel")

    cyclone = Equipment('CYCLONE', cyclone_rate, "Mixed", "Dust collectors", "Gas multi-cyclone", "321 stainless steel")

    # PSA
    psa_design = calculate_psa_design(df, p_ads_bar=15.0, p_des_bar=1.01, min_trains=2, n_beds_per_train=4)

    psa_shell_mass = psa_design["shell_mass_per_vessel"]
    psa_adsorbent_mass_total = psa_design["total_adsorbent_mass"]
    n_psa_vessels = psa_design["n_vessels"]

    # PSA vessels
    psa_vessel = Equipment(name="PSA-VESSEL", param=psa_shell_mass, process_type="Fluids", category="Pressure vessels", type="Vertical CS", material="Carbon steel", num_units=n_psa_vessels)

    # Zeolite 5A
    psa_adsorbent_volume_total_m3 = psa_adsorbent_mass_total / 795.0
    psa_adsorbent_volume_total_ft3 = psa_adsorbent_volume_total_m3 * 35.3146667

    psa_zeolite = Equipment(name="PSA-ZEOLITE-5A", param=psa_adsorbent_volume_total_ft3, process_type="Solids", category="Packings & adsorbents", type="Molecular sieves")

    # Complete PSA unit
    psa = CompositeEquipment(name="PSA", process_type="Fluids", components=[psa_vessel, psa_zeolite], category="Adsorption", type="Pressure-swing adsorber")

    NiMP_CSTR_equipment = [
        comp_1,
        comp_2, mtr_2,
        hx_1, hx_2,
        cooler_1, cooler_2,
        NiMP_CSTR_reactor,
        furnace, cyclone,
        psa
    ]


    # START OF PLANT
    methane_feed = df["CH4 Feed [kg/h]"].iloc[0]  # kg/h
    coolwater_total = df["Cool Water Total [kJ/s]"].iloc[0]
    carbon_prod = df["C Prod [kg/h]"].iloc[0]
    catalyst_flow = df["Cat Rate [kg/h]"].iloc[0]
    ch4_reactor_in = df["CH4 Reactor Feed [kmol/h]"].iloc[0]
    ch4_reactor_out = df["CH4 Reactor Outlet [kmol/h]"].iloc[0]

    residence_time = df["Cat Res. Time [s]"].iloc[0]
    deactivation_factor = df["Reactor Deactivation"].iloc[0]


    # Configuration for the NiMP plant.
    config = {
        'plant_name': 'CSTR',
        'process_type': 'Mixed',
        'country': country,
        'region': region,
        'equipment': NiMP_CSTR_equipment,
        'interest_rate': interest_rate,
        'tax_rate': tax_rate,
        'project_lifetime': project_lifetime,
        'plant_utilization': plant_utilization,

        'operator_hourly_rate': {
            'rate': operator_rate,
            'std': operator_std,
            'max': operator_max,
            'min': operator_min
        },

        'plant_products': {
            'hydrogen': {
                'production': h2_prod,
                'price': 5.0
            },

            'carbon': {
                'production': (carbon_prod + catalyst_flow) * 0.5 * 24,
                'price': 1,
                'std': 0.5,
                'max': 6,
                'min': 0
            }
        },

        'variable_opex_inputs': {
            'electricity': {
                'consumption': (mtr_1_pwr + mtr_2_pwr + electric_heating_power) * 24,
                'price': elecs_price,
                'std': elecs_std,
                'max': elecs_max,
                'min': elecs_min
            },

            'methane_feed': {
                'consumption': methane_feed * 24,
                'price': nat_gas_price,
                'std': nat_gas_std,
                'max': nat_gas_max,
                'min': nat_gas_min
            },

            'catalyst_feed': {
                'consumption': catalyst_flow * 24,
                'price': cat_price,
                'std': cat_std,
                'max': cat_max,
                'min': cat_min
            },

            'cooling_water': {
                'consumption': coolwater_total * 3600 * 24 / 1000,
                'price': cool_water_price,
                'std': cool_water_std,
                'max': cool_water_max,
                'min': cool_water_min
            },

            'carbon_disposal_fee': {
                'consumption': (carbon_prod + catalyst_flow) * 0.5 * 24,
                'price': 0.5,
                'std': 0.5 / 2,
                'max': 1,
                'min': 0
            }
        }
    }


    NiMP_CSTR_plant = Plant(config)

    ch4_conversion = (ch4_reactor_in - ch4_reactor_out) * 100 / ch4_reactor_in


    # START OF ECONOMIC ANALYSIS
    capex = NiMP_CSTR_plant.calculate_fixed_capital(print_results=False)
    isbl = NiMP_CSTR_plant.calculate_isbl(print_results=False)
    fixed_opex = NiMP_CSTR_plant.calculate_fixed_opex(print_results=False)
    variable_opex = NiMP_CSTR_plant.calculate_variable_opex(print_results=False)
    revenue = NiMP_CSTR_plant.calculate_revenue(print_results=False)

    cash_flow_styler = NiMP_CSTR_plant.calculate_cash_flow(print_results=True)
    cash_flow_df = cash_flow_styler.data

    npv = NiMP_CSTR_plant.calculate_npv(print_results=False)
    pbt = NiMP_CSTR_plant.calculate_payback_time(print_results=False)
    pbt_add = NiMP_CSTR_plant.calculate_payback_time(print_results=False, additional_capex=True)
    lcoh = NiMP_CSTR_plant.calculate_levelized_cost(print_results=False)
    roi = NiMP_CSTR_plant.calculate_roi(print_results=False)
    roi_add = NiMP_CSTR_plant.calculate_roi(print_results=False, additional_capex=True)
    irr = NiMP_CSTR_plant.calculate_irr(print_results=False)


    return {
        "plant_cstr": NiMP_CSTR_plant,
        "capex_cstr": capex,
        "isbl_cstr": isbl,
        "fixed_opex_cstr": fixed_opex,
        "variable_opex_cstr": variable_opex,
        "revenue_cstr": revenue,
        "cash_flow_cstr": cash_flow_df,
        "npv_cstr": npv,
        "pbt_cstr": pbt,
        "pbt_add_cstr": pbt_add,
        "lcoh_cstr": lcoh,
        "roi_cstr": roi,
        "roi_add_cstr": roi_add,
        "irr_cstr": irr,
        "catalyst_rate_cstr": catalyst_flow,
        "deac_factor_cstr": deactivation_factor,
        "conversion_cstr": ch4_conversion,
        "res_time_cstr": residence_time
    }

In [ ]:
#Changing Temperature; Fixed w (950kg) and a (0.2)
#H2
#PFR case study
csv_files_PFR_Tchange_DA02_W950 = {
    600: "../Results_Simulations/sim-PFR-100TPD-electric-t600-p1.07-c950-da0.2-psa15.csv",
}

# CSTR case study
csv_files_CSTR_Tchange_DA02_W950 = {
    600: "../Results_Simulations/sim-CSTR-100TPD-electric-t600-p1.07-c950-da0.2-psa15.csv",
}

In [ ]:
#Definition of arrays for each economic parameter
#PFR

temperatures = []
catalyst_amounts = []
deactivation_factors = []
pressures = []
capacity = []

plant_values_PFR = []
capex_values_PFR = []
isbl_values_PFR = []
fixed_opex_values_PFR = []
variable_opex_values_PFR = []
revenue_values_PFR = []
cash_flow_values_PFR = []
npv_values_PFR = []
pbt_values_PFR = []
pbt_add_values_PFR = []
lcoh_values_PFR = []
roi_values_PFR = []
roi_add_values_PFR = []
irr_values_PFR = []
catrate_PFR = []
deactivation_PFR = []
conversion_PFR = []
residence_time_PFR = []

for any_variable, path in csv_files_PFR_Tchange_DA02_W950.items():

    results = run_tea_pfr(path)
    temperatures.append(any_variable)
    #catalyst_amounts.append(any_variable)
    #deactivation_factors.append(any_variable)
    #pressures.append(any_variable)
    #capacity.append(any_variable)

    lcoh_values_PFR.append(results["lcoh_pfr"])

    plant_values_PFR.append(results["plant_pfr"])
    capex_values_PFR.append(results["capex_pfr"])
    isbl_values_PFR.append(results["isbl_pfr"])
    fixed_opex_values_PFR.append(results["fixed_opex_pfr"])
    variable_opex_values_PFR.append(results["variable_opex_pfr"])
    revenue_values_PFR.append(results["revenue_pfr"])
    cash_flow_values_PFR.append(results["cash_flow_pfr"])
    npv_values_PFR.append(results["npv_pfr"])
    pbt_values_PFR.append(results["pbt_pfr"])
    pbt_add_values_PFR.append(results["pbt_add_pfr"])
    roi_values_PFR.append(results["roi_pfr"])
    roi_add_values_PFR.append(results["roi_add_pfr"])
    irr_values_PFR.append(results["irr_pfr"])

    catrate_PFR.append(results["catalyst_rate_pfr"])
    deactivation_PFR.append(results["deac_factor_pfr"])
    conversion_PFR.append(results["conversion_pfr"])
    residence_time_PFR.append(results["res_time_pfr"])


In [ ]:
#Definition of arrays for each economic parameter
#CSTR

temperatures = []
catalyst_amounts = []
deactivation_factors = []
pressures = []
capacity = []

plant_values_CSTR = []
capex_values_CSTR = []
isbl_values_CSTR = []
fixed_opex_values_CSTR = []
variable_opex_values_CSTR = []
revenue_values_CSTR = []
cash_flow_values_CSTR = []
npv_values_CSTR = []
pbt_values_CSTR = []
pbt_add_values_CSTR = []
lcoh_values_CSTR = []
roi_values_CSTR = []
roi_add_values_CSTR = []
irr_values_CSTR = []
catrate_CSTR = []
deactivation_CSTR = []
conversion_CSTR = []
residence_time_CSTR = []

for any_variable, path in csv_files_CSTR_Tchange_DA02_W950.items():

    results = run_tea_cstr(path)
    temperatures.append(any_variable)
    #catalyst_amounts.append(any_variable)
    #deactivation_factors.append(any_variable)
    #pressures.append(any_variable)
    #capacity.append(any_variable)

    lcoh_values_CSTR.append(results["lcoh_cstr"])

    plant_values_CSTR.append(results["plant_cstr"])
    capex_values_CSTR.append(results["capex_cstr"])
    isbl_values_CSTR.append(results["isbl_cstr"])
    fixed_opex_values_CSTR.append(results["fixed_opex_cstr"])
    variable_opex_values_CSTR.append(results["variable_opex_cstr"])
    revenue_values_CSTR.append(results["revenue_cstr"])
    cash_flow_values_CSTR.append(results["cash_flow_cstr"])
    npv_values_CSTR.append(results["npv_cstr"])
    pbt_values_CSTR.append(results["pbt_cstr"])
    pbt_add_values_CSTR.append(results["pbt_add_cstr"])
    roi_values_CSTR.append(results["roi_cstr"])
    roi_add_values_CSTR.append(results["roi_add_cstr"])
    irr_values_CSTR.append(results["irr_cstr"])

    catrate_CSTR.append(results["catalyst_rate_cstr"])
    deactivation_CSTR.append(results["deac_factor_cstr"])
    conversion_CSTR.append(results["conversion_cstr"])
    residence_time_CSTR.append(results["res_time_cstr"])


In [ ]:
#Index to get single values
idx = temperatures.index(600)

In [ ]:
#Avg LCOH
print("CSTR:", lcoh_values_CSTR[idx])
print("PFR:", lcoh_values_PFR[idx])

avg_lcoh = (lcoh_values_CSTR[idx] + lcoh_values_PFR[idx]) / 2
print("AVG:", avg_lcoh)

# Save LCOH JSON ELECTRIC
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["FeMP", "ACMP", "NiMP"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Output directory
results_dir = project_root / "NiMP" / "Results_TEA"
results_dir.mkdir(parents=True, exist_ok=True)

lcoh_file = results_dir / "LCOH_heatings.json"

# Load existing results
if lcoh_file.exists():
    with open(lcoh_file, "r") as f:
        lcoh_results = json.load(f)
else:
    lcoh_results = {}

# Update results
lcoh_results["Electric"] = float(avg_lcoh)

# Save JSON
with open(lcoh_file, "w") as f:
    json.dump(lcoh_results, f, indent=4)

print(f"JSON saved in: {lcoh_file.resolve()}")

In [ ]:
# SPECIFIC CO2 EMISSIONS - ELECTRIC CASE

import pandas as pd

#Inputs
h2_production_daily = 100_000  # kg H2/day = 100 TPD
h2_production_hourly = h2_production_daily / 24  # kg H2/h

electricity_co2_factor = 0.283  # kg CO2/kWh

elec_mecc_eff = 0.90  # Motor + mechanical efficiency
react_heat_eff = 0.90  # Electric-heating efficiency

#Function for co2 emissions

def calculate_specific_co2_electric(csv_path):

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Direct emissions
    co2_direct_flow = df["CO2 Emission [kg/h]"].iloc[0] if "CO2 Emission [kg/h]" in df.columns else 0.0

    if pd.isna(co2_direct_flow):
        co2_direct_flow = 0.0

    # electricity consumption
    comp_1_pwr = df["Comp-1 Power [kW]"].iloc[0]
    comp_2_pwr = df["Comp-2 Power [kW]"].iloc[0]

    mtr_1_pwr = comp_1_pwr / elec_mecc_eff
    mtr_2_pwr = comp_2_pwr / elec_mecc_eff

    # electric heating
    heater_1_duty = df["Heater-1 Duty [MW]"].iloc[0]
    heater_2_duty = df["Heater-2 Duty [MW]"].iloc[0]
    reactor_duty = df["Reactor Duty [MW]"].iloc[0]

    total_heating = (
        heater_1_duty
        + heater_2_duty
        + reactor_duty
    ) * 1000  # kW thermal

    electric_heating_power = total_heating / react_heat_eff

    electricity_power = (
        mtr_1_pwr
        + mtr_2_pwr
        + electric_heating_power
    )

    # Indirect emissions
    co2_electric_flow = electricity_power * electricity_co2_factor

    # total emissions co2
    co2_total_flow = co2_direct_flow + co2_electric_flow

    # specific emissions co2
    co2_direct = co2_direct_flow / h2_production_hourly
    co2_electric = co2_electric_flow / h2_production_hourly
    co2_h2 = co2_total_flow / h2_production_hourly

    return {
        "h2_production_hourly": h2_production_hourly,
        "mtr_1_pwr": mtr_1_pwr,
        "mtr_2_pwr": mtr_2_pwr,
        "total_heating": total_heating,
        "electric_heating_power": electric_heating_power,
        "electricity_power": electricity_power,
        "co2_direct_flow": co2_direct_flow,
        "co2_electric_flow": co2_electric_flow,
        "co2_total_flow": co2_total_flow,
        "co2_direct": co2_direct,
        "co2_electric": co2_electric,
        "co2_h2": co2_h2
    }


def calculate_co2_dataset(csv_files):

    return {
        temperature: calculate_specific_co2_electric(path)
        for temperature, path in csv_files.items()
    }


co2_PFR_electric = calculate_co2_dataset(csv_files_PFR_Tchange_DA02_W950)
co2_CSTR_electric = calculate_co2_dataset(csv_files_CSTR_Tchange_DA02_W950)

selected_temperature = 600

pfr = co2_PFR_electric[selected_temperature]
cstr = co2_CSTR_electric[selected_temperature]

#Average
co2_direct_avg_electric = (pfr["co2_direct"] + cstr["co2_direct"]) / 2
co2_electric_avg_electric = (pfr["co2_electric"] + cstr["co2_electric"]) / 2
co2_h2_avg_electric = (pfr["co2_h2"] + cstr["co2_h2"]) / 2


# CSTR
print()
print(f"CSTR — {selected_temperature} °C")
print("-" * 75)

print(f"Electric heating power:         {cstr['electric_heating_power']:>12,.2f} kW")
print(f"Total electricity consumption:  {cstr['electricity_power']:>12,.2f} kW")
print(f"Direct CO2 emissions:           {cstr['co2_direct_flow']:>12,.4f} kg CO2/h")
print(f"Electricity CO2 emissions:      {cstr['co2_electric_flow']:>12,.4f} kg CO2/h")
print(f"Total CO2 emissions:            {cstr['co2_total_flow']:>12,.4f} kg CO2/h")

# PFR
print()
print(f"PFR — {selected_temperature} °C")
print("-" * 75)

print(f"Electric heating power:         {pfr['electric_heating_power']:>12,.2f} kW")
print(f"Total electricity consumption:  {pfr['electricity_power']:>12,.2f} kW")
print(f"Direct CO2 emissions:           {pfr['co2_direct_flow']:>12,.4f} kg CO2/h")
print(f"Electricity CO2 emissions:      {pfr['co2_electric_flow']:>12,.4f} kg CO2/h")
print(f"Total CO2 emissions:            {pfr['co2_total_flow']:>12,.4f} kg CO2/h")

# AVERAGE
print()
print("AVERAGE PFR/CSTR")
print("-" * 75)

print(f"Specific direct emissions:      {co2_direct_avg_electric:>12.6f} kg CO2/kg H2")
print(f"Specific electricity emissions: {co2_electric_avg_electric:>12.6f} kg CO2/kg H2")
print(f"TOTAL specific emissions:       {co2_h2_avg_electric:>12.6f} kg CO2/kg H2")

In [ ]:
# Save CO2 emissions JSON ELECTRIC
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["FeMP", "ACMP", "NiMP"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Output directory
results_dir = project_root / "NiMP" / "Results_TEA"
results_dir.mkdir(parents=True, exist_ok=True)

results_file = results_dir / "CO2_results.json"

# Load existing results
if results_file.exists():
    with open(results_file, "r") as f:
        co2_results = json.load(f)
else:
    co2_results = {}

# Update results
co2_results["Electric"] = co2_h2_avg_electric

# Save JSON
with open(results_file, "w") as f:
    json.dump(co2_results, f, indent=4)

print(f"JSON saved in: {results_file.resolve()}")